# Circular transmon builder

Build a circular-pad transmon qubit (`CircTransmon`) using qiskit-metal/SQDMetal, coupled to a feedline and to a quarter-wave meander readout resonator, plus a dedicated charge line for driving. Once the layout looks right, the design is saved to a `.json` file to be imported in simulations (e.g. PALACE).

In [ ]:
%load_ext autoreload
%autoreload 2

from collections import OrderedDict
from qiskit_metal import designs
from qiskit_metal import MetalGUI, Dict
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder

from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors

from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven #bonding
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround


from SQDMetal.Utilities.QUtilities import QUtilities
import matplotlib.pyplot as plt
import numpy as np
from qiskit_metal.toolbox_python.attr_dict import Dict

from toolkit.components.qubit.circle_transmon import CircTransmon
from toolkit.tools.comp_manipulation.adjust_design import adjust_design

from scipy.constants import c, e, hbar
from scipy.constants import physical_constants as pc
# %matplotlib inline

Helper functions: quarter-/half-wave length of a CPW resonator for a given effective permittivity and target frequency.

In [ ]:
res_len_lambda_2 = lambda eff, f: c/(2*np.sqrt(eff)*f)*1e3 # in mm
res_len_lambda_4 = lambda eff, f: c/(4*np.sqrt(eff)*f)*1e3 # in mm

Define the chip

In [ ]:
# design = designs.DesignPlanar({}, overwrite_enabled=True)

design = designs.DesignPlanar()
design.chips.main.material = 'silicon'

design.variables['cpw_width'] = '20um' #S from reference 2
design.variables['cpw_gap'] = '11um' #W from reference 2
design._chips['main']['size']['size_x'] = '5mm'
design._chips['main']['size']['size_y'] = '5mm'
design._chips['main']['size']['size_z'] = '-600um' #Our wafer thicknes is 525 um+/-10
design.chips.main.size.center_x = '1.25mm'
design.chips.main.size.center_y = '2.5mm'

# If you disable the next line with "overwrite_enabled", then you will need to
# delete a component [<component>.delete()] before recreating it.
design.overwrite_enabled = True

We also can open a GUI terminal to see the design

In [ ]:
gui = MetalGUI(design)

## Circular transmon qubit

Place a `CircTransmon` (circular-pad transmon) on the chip. Its `options` group into:
- **Position/orientation** and the shared CPW `cpw_width`/`cpw_gap`.
- **Pads** (`pad_radius`, `pad_gap`): the qubit's shunt capacitor.
- **`jj_options`**: the Josephson junction, either drawn directly or left as a gap (`jj_sim_gap`) for a   simulated/inserted JJ element when `export_mask=False`.
- **`claw_a_options` / `claw_b_options`**: up to two coupling claws, each placed on the `0`/`90`/`180`   arm, used to couple the qubit to a feedline or resonator. Here only `claw_a` is enabled   (`make_claw=True`); `claw_b` is disabled.
- **`charge_line`**: an optional dedicated pad+line for driving the qubit's charge, on a different arm   from the claws.

In [ ]:
options = Dict(
        # Position and orientation
        pos_x='1.8mm', pos_y='1.6mm', orientation=0, chip='main',
        cpw_width=design.variables['cpw_width'], cpw_gap=design.variables['cpw_gap'],

        # Pads (capacitor)
        pad_radius = '120um',
        pad_gap = '10um',

        # jj (mesoscopic: 0 or small junction: 1)
        jj_options=Dict(
            jj_width='5um',
            L_j = '40.7885nH',
            C_j = '0.72fF',
            export_mask = False,    # if True, draw a rectangle where the constriction will be made. If false, we keep a gap of jj_sim_gap in which we insert a jj element to make the simulation
            gds_cell_jj = 'gds_cell_jj',
            jj_sim_gap = '8um', # if we need a small junction, we can increase this gap
            ),
        
        # Coupled claws (2 claws allowed   )
        claw_a_options= Dict(
            make_claw=True,
            connector_location='90',  # 0 => 'west' arm, 90 => 'north' arm, 180 => 'east' arm
            claw_width='50um',
            claw_gap='30um',
            claw_length='200um',
            ground_spacing='10um',
            connector_length='70um',
            ),
        claw_b_options=Dict(
            make_claw=False,
            connector_location='0',  # 0 => 'west' arm, 90 => 'north' arm, 180 => 'east' arm
            claw_width='50um',
            claw_gap='30um',
            claw_length='100um',
            ground_spacing='10um',
            connector_length='50um',
            ),

        # Charge line
        charge_line=Dict(
            make_charge_line=True,
            location = '180', # 0 => 'west' arm, 90 => 'north' arm, 180 => 'east' arm. Must be different from claw_a and claw_b if they exist
            pad_radius='40um',
            pad_gap='40um', # gap between the pad and qubit
            ground_spacing='15um', # distance from the charge line to the ground
            line_length='100um',
            )
        )

# Create a fluxonium qubit with the Manucharyan design
qubit_1 = CircTransmon(design, 'Q1', options=options)


gui.rebuild()
gui.autoscale()

## Feedline

Two driven, wire-bonded launchpads (`LP1`, `LP2`) at the chip edge, joined by a straight transmission line. This is the main through-feedline and is not connected to the qubit.

In [ ]:
x1 = '0.25mm'
y1 = '3.2mm'

ops_1 = Dict(chip='main', pos_x=x1, pos_y=y1, orientation='360', lead_length='10um', 
             pad_width='120um', pad_gap='61um', trace_width=design.variables['cpw_width'] , trace_gap=design.variables['cpw_gap'] )
LP1 = LaunchpadWirebondDriven(design, 'LP1', options = ops_1)

# Driven Launchpad 2

x2 = '3mm'
y2 = '3.2mm'
ops_2 = Dict(chip='main', pos_x=x2, pos_y=y2, orientation='180', lead_length='10um',
             pad_width='120um', pad_gap='61um', trace_width=design.variables['cpw_width'] , trace_gap=design.variables['cpw_gap'] )
LP2 = LaunchpadWirebondDriven(design, 'LP2', options = ops_2)


# Rebuild the GUI 
# gui.rebuild()
# gui.autoscale()

## Feedline: connect LP1 to LP2 (not wired to the qubit)
opt_line = Dict(chip='main', trace_width =design.variables['cpw_width'] ,
        trace_gap =design.variables['cpw_gap'] , fillet='0um', hfss_wire_bonds = True, lead=Dict(end_straight='0.0mm'),
        pin_inputs=Dict(start_pin=Dict(component='LP1', pin='tie'),
                        end_pin=Dict(component='LP2', pin='tie')
                        )
               )

linea_1 = RoutePathfinder(design, 'linea_1', options = opt_line )

# Rebuild the GUI
gui.rebuild()
gui.autoscale() 


## Charge line launchpad

A third launchpad (`LPC`) that will be routed to the qubit's `charge_line` pin, so the qubit can be driven independently of the feedline/resonator.

In [ ]:
# charge lumped port
ops_charge_line = Dict(chip='main', pos_x='0.5mm', pos_y='1.6mm', orientation='0', lead_length='10um',
             pad_width='120um', pad_gap='61um', trace_width=design.variables['cpw_width'] , trace_gap=design.variables['cpw_gap'] )

LPC = LaunchpadWirebondDriven(design, 'LPC', options = ops_charge_line)

# Rebuild the GUI
gui.rebuild()
gui.autoscale()


## Connect the charge line to the qubit

In [ ]:
# Connect LPC with charge_line pad
# anchors_points = [(0.7, 2), (0.7, 2.5), (0.8, 2.5), (1, 1.6)]
# anchors_dict = OrderedDict({i: point for i, point in enumerate(anchors_points)})

opt_line_charge = Dict(chip='main', trace_width =design.variables['cpw_width'],
        trace_gap = design.variables['cpw_gap'], hfss_wire_bonds = True, lead=Dict(end_straight='0.0mm'),
        pin_inputs=Dict(start_pin=Dict(component='LPC', pin='tie'),
                        end_pin=Dict(component='Q1', pin='charge_line')
                        )
                )

con_charge_line = RoutePathfinder(design, 'con_charge_line', options = opt_line_charge )
gui.rebuild()
gui.autoscale()

## Set STG for the readout resonator

`RouteMeander` (like any `QRoute`) needs a start and an end pin. The end pin is the qubit's `claw_a` pin; the start pin is provided by a `ShortToGround` (`stg1`), which just terminates a CPW to ground at a chosen position/orientation. This cell also estimates the target physical length of the resonator for the desired frequency.

In [ ]:
## Estimation for the resonator length
eff=(11.9+1)/2 ## Si and Air
f=7e9  # Resonator frequency in Hz

stg1 = ShortToGround(design, 'stg1', options=Dict(chip='main', pos_x='1mm',  pos_y='2.3mm', orientation='180'))

## Set meander route

Route the quarter-wave meander resonator between `stg1` (shorted end) and the qubit's `claw_a` pin (coupled end).

In [ ]:
jogs = OrderedDict()
jogs[0] = ["L", '400um']
jogs[1] = ["L", '100um']
# jogs[1] = ["L", '300um']
# jogs[2] = ["R", '200um']
l = res_len_lambda_4(eff, f)  # in mm
total_length = f'{l}mm'

opt_meander = Dict(
    trace_width= design.variables['cpw_width'] ,
    trace_gap= design.variables['cpw_gap'] ,
    hfss_wire_bonds= True,
    meander=Dict(
        spacing= '250um',
        asymmetry= '0um'),    
    total_length= total_length,
    fillet = '90um',
    pin_inputs=Dict(
        end_pin=Dict(
            component= 'Q1',
            pin= 'claw_a'),
        start_pin=Dict(
            component= 'stg1',
            pin= 'short')),
    lead=Dict(
        start_straight='0.02mm',
        end_straight='1.278mm',
        end_jogged_extension=jogs
        ),
)
meandro_1 = RouteMeander(design, 'meandro_1',  options=opt_meander)

# # rebuild the GUI
gui.rebuild()
gui.autoscale()

## Adjust chip bounding box

Shrink the chip to the real bounding box of the drawn circuit before exporting, so the simulation doesn't waste mesh/compute on empty substrate.

In [ ]:
# Shrink the chip to the real bounding box of the drawn circuit before exporting to PALACE.

adjust_design(design)
gui.rebuild()
gui.autoscale()

## Save design

Qiskit Metal doesn't have a working built-in way to save a design (there's a function, but it doesn't work, and its own docstring lists it as a ToDo). There's also a function to dump the design as a Python script, but that's not convenient either. Instead, use the `save_design`/`load_design` helpers from `toolkit/tools/sim/save_load_design.py`: `save_design` writes the design to a `.json` file and `load_design` reads it back.

In [ ]:
from toolkit.tools.sim.save_load_design import save_design, load_design

In [ ]:
path2save = "circTransmon.json"

save_design(design, path2save)